# Fundamentos y Ventajas de la clase `std::vector`
## El Modelo de Memoria y la Abstracción de Datos
La clase `std::vector` se define como un Contenedor de Secuencia con Almacenamiento Contiguo. A diferencia de los arreglos estáticos, el vector implementa una gestión dinámica mediante el uso de tres componentes críticos en su estructura interna:
* **Puntero de Inicio (First)**: Apunta a la base del bloque asignado en el heap.
* **Puntero de Finalización (Last)**: Marca el final de los elementos inicializados (determina el size).
* **Puntero de Capacidad (End_of_storage)**: Marca el límite de la memoria reservada (determina el capacity).

## Análisis de Complejidad y Rendimiento

La eficiencia no es solo "rapidez", sino el cumplimiento de cotas asintóticas:
* **Acceso Aleatorio**: $O(1)$ debido a que la dirección de cualquier elemento $i$ se calcula como $Base + (i \times sizeof(T))$.
* **Inserción Amortizada**: El costo de push_back es $O(1)$ amortizado. Cuando `size == capacity`, el vector realiza una reasignación (usualmente un factor de $1.5$ o $2$). El costo de copiar $n$ elementos es compensado por las siguientes $n$ inserciones de costo constante.
* **Localidad de Datos**: Al ser memoria contigua, el vector maximiza el aprovechamiento de la jerarquía de memoria (L1/L2/L3 cache), reduciendo los cache misses en algoritmos de alto cómputo.

## Semántica de Copia y Movimiento
La clase std::vector cumple con la Regla de los Cinco. Para optimizar procesos algorítmicos, es fundamental distinguir entre:
* **Copy Semantics**: Crea una copia profunda del bloque de memoria ($O(n)$).
* **Move Semantics (C++11)**: Transfiere la propiedad de los punteros internos ($O(1)$), evitando duplicidad innecesaria de datos en el heap.

### Ejemplo 1

Este código permite observar empíricamente la política de crecimiento exponencial y la invalidación de punteros/iteradores, un tema recurrente en el libro.

In [1]:
#include <iostream>
#include <vector>

In [2]:
std::vector<int> v;
size_t capacidad_inicial = v.capacity();

std::cout << "Capacidad inicial: " << capacidad_inicial << std::endl;

for(int i = 0; i < 100; ++i) {
    v.push_back(i);
    if (v.capacity() != capacidad_inicial) {
        capacidad_inicial = v.capacity();
        // Esto demuestra el crecimiento exponencial (factor de carga)
        std::cout << "Reasignación detectada. Nueva capacidad: " << capacidad_inicial 
                      << " | Tamaño: " << v.size() << std::endl;
    }
}

Capacidad inicial: 0
Reasignación detectada. Nueva capacidad: 1 | Tamaño: 1
Reasignación detectada. Nueva capacidad: 2 | Tamaño: 2
Reasignación detectada. Nueva capacidad: 4 | Tamaño: 3
Reasignación detectada. Nueva capacidad: 8 | Tamaño: 5
Reasignación detectada. Nueva capacidad: 16 | Tamaño: 9
Reasignación detectada. Nueva capacidad: 32 | Tamaño: 17
Reasignación detectada. Nueva capacidad: 64 | Tamaño: 33
Reasignación detectada. Nueva capacidad: 128 | Tamaño: 65


### Ejemplo 2

Uso de `emplace_back` vs `push_back` para entender la optimización en el paso de argumentos y la construcción in-place, evitando llamadas innecesarias al constructor de copia.

In [3]:
#include <string>

class Analizador {
private:
    std::string nombre;
    double* datos;

public:
    // Constructor con parámetros
    Analizador(std::string n) : nombre(n) {
        datos = new double[10]; // Simulamos carga de datos
        std::cout << "[C] Constructor para '" << nombre 
                  << "' en STACK: " << this 
                  << " | HEAP asignado: " << static_cast<void*>(datos) << std::endl;
    }

    // Constructor de Movimiento
    Analizador(Analizador&& otro) noexcept : nombre(std::move(otro.nombre)), datos(otro.datos) {
        otro.datos = nullptr;
        std::cout << "[M] MOVIMIENTO de '" << nombre 
                  << "'. Nueva dirección STACK: " << this 
                  << " | Mantiene HEAP: " << static_cast<void*>(datos) << std::endl;
    }

    // Destructor
    ~Analizador() {
        if (datos != nullptr) {
            std::cout << "[D] Liberando HEAP de '" << nombre << "' en: " << static_cast<void*>(datos) << std::endl;
            delete[] datos;
        }
    }

    // Deshabilitamos copia para obligar a ver la diferencia
    Analizador(const Analizador&) = delete;
};

void comparativa_optimizacion() {
    std::vector<Analizador> v2;
    v2.reserve(5); // Reservamos para evitar que las reasignaciones ensucien la salida

    std::cout << "--- ESCENARIO 1: push_back ---" << std::endl;
    // 1. Se crea un objeto temporal en el stack
    // 2. Se mueve al vector
    // 3. El temporal se destruye
    v2.push_back(Analizador("Temp_Push"));

    std::cout << "\n--- ESCENARIO 2: emplace_back ---" << std::endl;
    // Solo se llama al constructor una vez, directamente en la memoria del vector.
    // No hay temporales, no hay movimientos, no hay destrucción extra.
    v2.emplace_back("InPlace_Emplace");

    std::cout << "\n--- FIN DE FUNCIÓN (Limpieza final) ---" << std::endl;
}

comparativa_optimizacion();

--- ESCENARIO 1: push_back ---
[C] Constructor para 'Temp_Push' en STACK: 0xfffff0c731d0 | HEAP asignado: 0xaaaaed805200
[M] MOVIMIENTO de 'Temp_Push'. Nueva dirección STACK: 0xaaaaedb85720 | Mantiene HEAP: 0xaaaaed805200

--- ESCENARIO 2: emplace_back ---
[C] Constructor para 'InPlace_Emplace' en STACK: 0xaaaaedb85748 | HEAP asignado: 0xaaaaeda6cdf0

--- FIN DE FUNCIÓN (Limpieza final) ---
[D] Liberando HEAP de 'Temp_Push' en: 0xaaaaed805200
[D] Liberando HEAP de 'InPlace_Emplace' en: 0xaaaaeda6cdf0


# Navegación con Iteradores
## La Abstracción del Iterador: 
Un iterador es un objeto que emula el comportamiento de un puntero. Es un mapeo entre un conjunto de índices y un espacio de memoria. El `std::vector` implementa Random Access Iterators (Iteradores de Acceso Aleatorio). Esta es la categoría más alta en la jerarquía de la STL porque permite:
* Aritmética de punteros: `it + n` o `it - m` en tiempo constante $$O(1)$$.
* Comparaciones: `it1 < it2` para saber la posición relativa.
* Desreferenciación: `*it` para acceder al valor.

## El Rango Semiebierto $[begin, end)$
La STL utiliza rangos donde el inicio es inclusivo y el final es exclusivo:
* `begin()`: Apunta al primer elemento.
* `end()`: Apunta a la posición de memoria inmediatamente posterior al último elemento.

**Nota técnica**: Nunca se debe desreferenciar `end()`. Su propósito es servir como centinela para detener ciclos.

## Const-Correctness (cbegin, cend)
Para un desarrollador profesional, usar `cbegin()` y `cend()` es una garantía contractual. Le dice al compilador (y a otros programadores) que el algoritmo solo lee los datos. Esto permite optimizaciones de solo lectura a nivel de registro y evita efectos secundarios no deseados.

### Ejemplo 3: Aritmética de Iteradores y Direcciones de Memoria

Este ejemplo muestra cómo el iterador "salta" en la memoria del hardware de forma proporcional al tamaño del tipo de dato.

In [4]:
void demostracion_itinerarios() {
    std::vector<int> v3 = {10, 20, 30, 40, 50};
    
    // Obtenemos iteradores
    auto it_inicio = v3.begin();
    auto it_final  = v3.end();

    std::cout << "--- Geometría del Vector ---" << std::endl;
    std::cout << "Dirección begin() : " << &(*it_inicio) << " (Valor: " << *it_inicio << ")" << std::endl;
    std::cout << "Dirección end()   : " << &(*it_final)  << " (¡Cuidado! Apunta fuera)" << std::endl;

    // Aritmética de Acceso Aleatorio O(1)
    auto it_medio = it_inicio + 2; // Salto matemático de 2 posiciones
    
    std::cout << "\n--- Aritmética de Iteradores ---" << std::endl;
    std::cout << "it_inicio + 2     : " << &(*it_medio) << " (Valor: " << *it_medio << ")" << std::endl;
    
    // Distancia entre iteradores (Matemáticamente: it2 - it1)
    std::cout << "Distancia total   : " << (it_final - it_inicio) << " elementos." << std::endl;
}

demostracion_itinerarios();

--- Geometría del Vector ---
Dirección begin() : 0xaaaaedaa8f60 (Valor: 10)
Dirección end()   : 0xaaaaedaa8f74 (¡Cuidado! Apunta fuera)

--- Aritmética de Iteradores ---
it_inicio + 2     : 0xaaaaedaa8f68 (Valor: 30)
Distancia total   : 5 elementos.


### Ejemplo 4: El Peligro de la "Invalidación de Iteradores"

Muestra cómo un iterador se convierte en un puntero colgante (dangling pointer) cuando el vector crece y se mueve en el heap.

In [5]:
void demostracion_invalidacion() {
    std::vector<int> v = {100, 200, 300};
    
    // 1. Estado Inicial
    std::cout << "--- 1. ESTADO INICIAL ---" << std::endl;
    std::cout << "Capacidad actual: " << v.capacity() << std::endl;
    std::cout << "Dirección real de v[0]: " << &v[0] << std::endl;

    // Guardamos un iterador al inicio
    auto it_viejo = v.begin();
    std::cout << "Iterador 'it_viejo' apunta a: " << &(*it_viejo) 
              << " | Valor: " << *it_viejo << std::endl;

    // 2. Provocar la "Mudanza" (Realloc)
    // Agregamos elementos hasta que supere la capacidad de 3
    std::cout << "\n--- 2. AGREGANDO ELEMENTOS (Provocando Reasignación) ---" << std::endl;
    for(int i = 0; i < 5; ++i) v.push_back(i);

    // 3. Verificación de la Catástrofe
    std::cout << "\n--- 3. RESULTADO DE LA MUDANZA ---" << std::endl;
    std::cout << "Nueva Capacidad: " << v.capacity() << std::endl;
    std::cout << "Nueva dirección real de v[0]: " << &v[0] << std::endl;

    std::cout << "\n--- 4. COMPARATIVA CRÍTICA ---" << std::endl;
    std::cout << "El iterador 'it_viejo' se quedó en: " << &(*it_viejo) << std::endl;
    
    if (&(*it_viejo) != &v[0]) {
        std::cout << "Valor de v[0]: " << v[0] << std::endl;
        std::cout << "[ALERTA] El iterador es INVÁLIDO. " << std::endl;
        std::cout << "Iterador 'it_viejo' apunta a: " << &(*it_viejo) 
              << " | Valor con 'it_viejo': " << *it_viejo << std::endl;
        std::cout << "Apunta a memoria que el SO ya liberó o marcó como basura." << std::endl;
        // Acceder a *it_viejo aquí es comportamiento indefinido (UB)
    }
}

demostracion_invalidacion();

--- 1. ESTADO INICIAL ---
Capacidad actual: 3
Dirección real de v[0]: 0xaaaaedb778b0
Iterador 'it_viejo' apunta a: 0xaaaaedb778b0 | Valor: 100

--- 2. AGREGANDO ELEMENTOS (Provocando Reasignación) ---

--- 3. RESULTADO DE LA MUDANZA ---
Nueva Capacidad: 12
Nueva dirección real de v[0]: 0xaaaaed98fd40

--- 4. COMPARATIVA CRÍTICA ---
El iterador 'it_viejo' se quedó en: 0xaaaaedb778b0
Valor de v[0]: 100
[ALERTA] El iterador es INVÁLIDO. 
Iterador 'it_viejo' apunta a: 0xaaaaedb778b0 | Valor con 'it_viejo': -307552768
Apunta a memoria que el SO ya liberó o marcó como basura.


# Gestión de Memoria

## Capacidad vs. Tamaño  
* `size(n)`: Cuántos elementos hay. Afecta el tiempo de ejecución de algoritmos como std::sort.
* `capacity(C)`: Cuánta memoria hay reservada. Afecta cuántas inserciones $O(1)$ podemos hacer antes de una costosa reasignación $O(n)$

## Estrategias de Optimización:
* `reserve(n)`: Es una directiva para el asignador de memoria. Si sabemos que leeremos un archivo de 1 millón de datos, `reserve(1000000)` elimina las ~20 reasignaciones intermedias que el vector haría por defecto
* `shrink_to_fit()`: Es lo opuesto. Si un vector tenía 1 millón de elementos y borramos 999,999, el vector sigue ocupando el espacio de 1 millón. `shrink_to_fit()` devuelve esa memoria sobrante al sistema operativo.

### Ejemplo 5: El Costo del Crecimiento Dinámico

Este ejemplo permite cuantificar cuántas veces se "mueve" un vector si no somos cuidadosos.

In [6]:
#include <iostream>
#include <vector>
#include <chrono>

void benchmark_memoria() {
    const int N = 10000000; // 10 Millones de elementos
    
    std::cout << "=== ESCENARIO A: SIN RESERVE (Crecimiento Dinámico) ===" << std::endl;
    
    auto inicio1 = std::chrono::high_resolution_clock::now();
    
    std::vector<int> v_dinamico;
    int reasignaciones = 0;
    size_t cap_previa = 0;

    for(int i = 0; i < N; ++i) {
        v_dinamico.push_back(i);
        
        // Detectamos cuando el vector "se muda" en el Heap
        if (v_dinamico.capacity() != cap_previa) {
            cap_previa = v_dinamico.capacity();
            reasignaciones++;
            
            // Imprimimos solo las primeras para no saturar el notebook, 
            // pero lo suficiente para ver el patrón.
            if (reasignaciones <= 10 || i > N - 100) {
                std::cout << "[MUDANZA #" << reasignaciones 
                          << "] Size: " << i 
                          << " | Nueva Capacidad: " << cap_previa 
                          << " | Nueva Dirección: " << &v_dinamico[0] << std::endl;
            }
        }
    }
    
    auto fin1 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> ms1 = fin1 - inicio1;

    std::cout << "\n=== ESCENARIO B: CON RESERVE (Asignación Única) ===" << std::endl;
    
    auto inicio2 = std::chrono::high_resolution_clock::now();
    
    std::vector<int> v_optimizado;
    v_optimizado.reserve(N); // Una sola petición al SO
    
    std::cout << "[SISTEMA] Capacidad inicial fija: " << v_optimizado.capacity() 
              << " | Dirección: " << &v_optimizado[0] << std::endl;

    for(int i = 0; i < N; ++i) {
        v_optimizado.push_back(i);
    }
    
    auto fin2 = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> ms2 = fin2 - inicio2;

    std::cout << "\n--- COMPARATIVA FINAL ---" << std::endl;
    std::cout << ">> Mudanzas en A: " << reasignaciones << " veces." << std::endl;
    std::cout << ">> Tiempo en A:   " << ms1.count() << " ms" << std::endl;
    std::cout << ">> Tiempo en B:   " << ms2.count() << " ms" << std::endl;
    std::cout << ">> ACELERACIÓN:  " << (ms1.count() / ms2.count()) << "x más rápido." << std::endl;
}

benchmark_memoria();

=== ESCENARIO A: SIN RESERVE (Crecimiento Dinámico) ===
[MUDANZA #1] Size: 0 | Nueva Capacidad: 1 | Nueva Dirección: 0xaaaaedb71ea0
[MUDANZA #2] Size: 1 | Nueva Capacidad: 2 | Nueva Dirección: 0xaaaaecac0fa0
[MUDANZA #3] Size: 2 | Nueva Capacidad: 4 | Nueva Dirección: 0xaaaaedb71ea0
[MUDANZA #4] Size: 4 | Nueva Capacidad: 8 | Nueva Dirección: 0xaaaaed0ef980
[MUDANZA #5] Size: 8 | Nueva Capacidad: 16 | Nueva Dirección: 0xaaaaedc64f30
[MUDANZA #6] Size: 16 | Nueva Capacidad: 32 | Nueva Dirección: 0xaaaaedd25c90
[MUDANZA #7] Size: 32 | Nueva Capacidad: 64 | Nueva Dirección: 0xaaaaed985a00
[MUDANZA #8] Size: 64 | Nueva Capacidad: 128 | Nueva Dirección: 0xaaaaedb76c40
[MUDANZA #9] Size: 128 | Nueva Capacidad: 256 | Nueva Dirección: 0xaaaaede58680
[MUDANZA #10] Size: 256 | Nueva Capacidad: 512 | Nueva Dirección: 0xaaaaedd37ab0

=== ESCENARIO B: CON RESERVE (Asignación Única) ===
[SISTEMA] Capacidad inicial fija: 10000000 | Dirección: 0xffff9d9da010

--- COMPARATIVA FINAL ---
>> Mudanzas en A

### Ejemplo 6: `resize` vs `reserve` (Impacto en el Ciclo de Vida)
Este ejemplo permite visualizar la diferencia entre simplemente apartar memoria y construir físicamente objetos en el Heap mediante el uso de una clase testigo.

In [7]:
class Elemento {
public:
    double coordenadas[3]; // El objeto tiene un "cuerpo" (24 bytes)
    int id;

    // Constructor: Se activa con resize()
    Elemento() : id(10) {
        std::cout << "  [C] Constructor invocado en: " << this << " (Tamaño: " << sizeof(*this) << " bytes)" << std::endl;
    }

    ~Elemento() {
        std::cout << "  [D] Destructor invocado en:  " << this << std::endl;
    }

    // Declaración 'friend' DENTRO de la clase
    // Esto permite definirlo aquí mismo, pero C++ lo trata como función global
    friend std::ostream& operator<<(std::ostream& os, const Elemento& e) {
    os << "Elemento[ID: " << e.id << "]";
    return os;
}
};

void demostracion_tecnica() {
    std::cout << "--- ESCENARIO A: reserve(3) ---" << std::endl;
    std::vector<Elemento> v_a;
    v_a.reserve(3); 
    std::cout << "Size: " << v_a.size() << " | Capacity: " << v_a.capacity() << std::endl;
    std::cout << "Nota: El contador de objetos es 0. El Heap tiene el espacio, pero no hay objetos vivos." << std::endl;

    std::cout << "\n--- ESCENARIO B: resize(3) ---" << std::endl;
    std::vector<Elemento> v_b;
    v_b.resize(3);
    std::cout << "ID del objeto: " << v_b[0] << std::endl;
    std::cout << "ID del objeto: " << v_b[1] << std::endl;
    std::cout << "ID del objeto: " << v_b[2] << std::endl;
    
    std::cout << "Size: " << v_b.size() << " | Capacity: " << v_b.capacity() << std::endl;
    
    std::cout << "\n--- ANÁLISIS DE DIRECCIONES (CONTIGÜIDAD) ---" << std::endl;
    for(int i = 0; i < 3; ++i) {
        std::cout << "Objeto [" << i << "] dirección: " << &v_b[i] << std::endl;
    }
}

demostracion_tecnica();

--- ESCENARIO A: reserve(3) ---
Size: 0 | Capacity: 3
Nota: El contador de objetos es 0. El Heap tiene el espacio, pero no hay objetos vivos.

--- ESCENARIO B: resize(3) ---
  [C] Constructor invocado en: 0xaaaaee0bc9d0 (Tamaño: 32 bytes)
  [C] Constructor invocado en: 0xaaaaee0bc9f0 (Tamaño: 32 bytes)
  [C] Constructor invocado en: 0xaaaaee0bca10 (Tamaño: 32 bytes)
ID del objeto: Elemento[ID: 10]
ID del objeto: Elemento[ID: 10]
ID del objeto: Elemento[ID: 10]
Size: 3 | Capacity: 3

--- ANÁLISIS DE DIRECCIONES (CONTIGÜIDAD) ---
Objeto [0] dirección: 0xaaaaee0bc9d0
Objeto [1] dirección: 0xaaaaee0bc9f0
Objeto [2] dirección: 0xaaaaee0bca10
  [D] Destructor invocado en:  0xaaaaee0bc9d0
  [D] Destructor invocado en:  0xaaaaee0bc9f0
  [D] Destructor invocado en:  0xaaaaee0bca10
